# Submission 07 — Ensemble multi-task (mDeBERTa-v3 + XLM-R)

**Grupo 11 — Entrega 2 (Deep Learning) — MODELO FINAL (DL puro)**

## Hipótesis
Sobre el baseline limpio de NB09, tres mejoras compuestas:

1. **Ensemble de dos encoders complementarios y estables** — `FacebookAI/xlm-roberta-base`
   (A, multilingüe: cubre el latín/italiano/catalán del corpus) + `dccuchile/bert-base-spanish-wwm-cased`
   (B, BETO, español-específico). Multi vs mono, RoBERTa vs BERT, SentencePiece vs WordPiece →
   errores descorrelacionados. *(mDeBERTa-v3 sería un 3er multilingüe ideal pero diverge a NaN;
   se intenta aparte en NB10.)*
2. **Multi-task siglo + década** (α=0.10): la cabeza de siglo (4 clases, fácil) estabiliza
   la representación y reduce errores inter-siglo.
3. **Datos externos** (toggle `USE_EXTERNAL_DATA`, OFF por defecto): párrafos con fecha de
   Gutenberg/Wikisource, capados al 20% del train, priorizando décadas tempranas.

## Receta corregida (vs el viejo NB07)
- **CE plana, NO smoothing gaussiano** (el smoothing bajó el F1 en NB06: 0.24→0.20).
- **XLM-R y BETO en fp16** (ambos estables; guardia NaN automática en `dl_utils`).
- **max_length=384** + ventana deslizante en inferencia.
- **Modelos vigentes y multilingües** (descartados bertin/BNE: deprecados / bug LayerNorm).
- Logging detallado por paso (loss/lr/gnorm/throughput/ETA).

## F1-macro esperado: +0.01 a +0.03 sobre NB09 single

> Nota de cómputo: entrenar 2 modelos (~3-4.5h en T4). Cabe en el límite de 12h de Kaggle.


## 0. Setup

Sube `utils.py` y `dl_utils.py` como Kaggle Datasets (`utils-py`, `dl-utils`).


In [1]:
# %pip install -q transformers==4.45.* datasets accelerate safetensors sentencepiece scikit-learn joblib

import os, sys, warnings, json, random
warnings.filterwarnings("ignore")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

for p in [
    os.getcwd(),
    "/kaggle/input/datasets/diegomolanoroa/utils-py",
    "/kaggle/input/datasets/diegomolanoroa/dl-utils",
    "/kaggle/input/utils-py",
    "/kaggle/input/dl-utils",
]:
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)

import utils, dl_utils
utils.set_global_seed(utils.SEED)

import torch
print("CUDA:", torch.cuda.is_available(),
      "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")


2026-05-20 18:01:55.918420: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779300116.247996      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779300116.346673      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779300117.172527      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779300117.172572      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779300117.172576      24 computation_placer.cc:177] computation placer alr

CUDA: True | device: Tesla T4


In [2]:
MODEL_NAME_A = "FacebookAI/xlm-roberta-base"             # multilingüe, estable
MODEL_NAME_B = "dccuchile/bert-base-spanish-wwm-cased"   # BETO, español-específico
MAX_LENGTH = 384
ALPHA_CENTURY = 0.10

# --- Regularización (palancas contra el overfitting medido en NB09) ---------
# NB09 mostró overfitting: train_loss seguía cayendo (3.18->1.18) pero val F1 se
# aplanó en ~ep10 (0.285). El cuello de botella es generalización, no capacidad.
# Dos palancas cableadas, en su valor NEUTRO por defecto, para que la primera
# corrida sea el ensemble LIMPIO comparable 1:1 con NB09 (0.29130 Kaggle):
DROPOUT = 0.1                  # subir a 0.2-0.3 para el ablation de regularización
# Smoothing UNIFORME (≠ gaussiano de NB06): regularizador estándar que NO
# distorsiona la estructura ordinal (no premia décadas vecinas). Flip a True
# para el ablation; si NO sube el F1 en holdout, dejarlo OFF.
USE_UNIFORM_SMOOTH = False
SMOOTH_EPS = 0.05
LOSS_KIND = "uniform" if USE_UNIFORM_SMOOTH else "ce"

# Augmentación realista (NO ligaduras) — OFF por defecto; la de NB05 no ayudó
USE_AUGMENT = False
P_SPACE, P_HYPHEN, P_CHAR = 0.05, 0.015, 0.01

# Datos externos (§5.1) — OFF hasta validar el CSV (ver scripts/fetch_external.py)
USE_EXTERNAL_DATA = False
EXTERNAL_CSV = "/kaggle/input/datasets/diegomolanoroa/textos-externos/textos_externos.csv"
if not os.path.exists(EXTERNAL_CSV):
    EXTERNAL_CSV = "../../data_external/textos_externos.csv"

# XLM-R y BETO: ambos estables en fp16 (más rápido). Misma receta para los dos.
cfg_a = dl_utils.TrainConfig(max_length=MAX_LENGTH, batch_size=16, grad_accum=2, lr=2e-5,
                             epochs=10, loss_kind=LOSS_KIND, smooth_eps=SMOOTH_EPS,
                             alpha_century=ALPHA_CENTURY, precision="fp16",
                             early_stopping_patience=3, log_every=50)
cfg_b = dl_utils.TrainConfig(max_length=MAX_LENGTH, batch_size=16, grad_accum=2, lr=2e-5,
                             epochs=10, loss_kind=LOSS_KIND, smooth_eps=SMOOTH_EPS,
                             alpha_century=ALPHA_CENTURY, precision="fp16",
                             early_stopping_patience=3, log_every=50)

DATA_DIR = "../../data"
for cand in [
    "/kaggle/input/competitions/parte-2-competencia-aprendizaje-de-maquina-2026-10",
    "/kaggle/input/aprendizaje-maquina-2026-10-parte-2",
    "../../data",
]:
    if os.path.exists(cand):
        DATA_DIR = cand
        break
OUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
print("DATA_DIR:", DATA_DIR, "| OUT_DIR:", OUT_DIR, "| externos:", USE_EXTERNAL_DATA)
print(f"Regularización -> dropout={DROPOUT} | loss={LOSS_KIND}"
      + (f" (eps={SMOOTH_EPS})" if LOSS_KIND == "uniform" else "") + f" | augment={USE_AUGMENT}")


DATA_DIR: /kaggle/input/competitions/parte-2-competencia-aprendizaje-de-maquina-2026-10 | OUT_DIR: /kaggle/working | externos: False
Regularización -> dropout=0.1 | loss=ce | augment=False


## 1. Datos (interno + externo opcional) + índices década/siglo

In [3]:
import numpy as np
import pandas as pd
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split

corpus = utils.load_corpus(DATA_DIR)
utils.quick_summary(corpus)

train_df = corpus.train[["text", "text_clean", "decade"]].copy()
train_df["source"] = "internal"

if USE_EXTERNAL_DATA and os.path.exists(EXTERNAL_CSV):
    ext = utils.load_external_data(EXTERNAL_CSV, base_n=len(corpus.train), cap_frac=0.20, seed=utils.SEED)
    print("Externos cargados:", len(ext), "| por fuente:", ext["source"].value_counts().to_dict())
    train_df = pd.concat([train_df, ext[["text", "text_clean", "decade", "source"]]], ignore_index=True)
else:
    print("Sin datos externos (toggle OFF o CSV ausente)")
print("Train final:", len(train_df), "| fuentes:", train_df["source"].value_counts().to_dict())

y_dec = np.array([utils.DECADE_TO_IDX[int(d)] for d in train_df["decade"].values])
y_cen = np.array([utils.CENTURY_TO_IDX[utils.decade_to_century(int(d))] for d in train_df["decade"].values])
texts = train_df["text_clean"].values
idx_tr, idx_va = train_test_split(np.arange(len(train_df)), test_size=0.10,
                                  stratify=y_dec, random_state=utils.SEED)
print("Train:", len(idx_tr), "| Val:", len(idx_va))


def augment_fn(text):
    return utils.augment_realistic(text, rng=random, p_space=P_SPACE, p_hyphen=P_HYPHEN, p_char=P_CHAR)
AUG = augment_fn if USE_AUGMENT else None


Train shape: (31403, 3) | Eval shape: (3490, 3)
Décadas únicas: 39
Rango décadas: 150 - 188
Ejemplos por clase  min/mean/max: 754 / 805.2 / 848
Palabras por texto  median/p95/max: 50 / 255 / 1146
Sin datos externos (toggle OFF o CSV ausente)
Train final: 31403 | fuentes: {'internal': 31403}
Train: 28262 | Val: 3141


## 2. Modelo A — XLM-R multi-task (fp16)

In [4]:
tok_a = AutoTokenizer.from_pretrained(MODEL_NAME_A, use_fast=True)
model_a = dl_utils.MultiTaskClassifier(MODEL_NAME_A, use_century=True, dropout=DROPOUT)
print("A:", MODEL_NAME_A, "|", round(sum(p.numel() for p in model_a.parameters())/1e6, 1), "M")


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


A: FacebookAI/xlm-roberta-base | 278.1 M


In [5]:
%%time
model_a, hist_a, f1_a = dl_utils.train_model(
    model_a, texts[idx_tr], y_dec[idx_tr], y_cen[idx_tr],
    texts[idx_va], y_dec[idx_va], y_cen[idx_va],
    tok_a, cfg=cfg_a, augment_fn=AUG, label="A-xlmr",
)
print("\nBest val F1 (A):", round(f1_a, 4))


[A-xlmr] precisión=fp16 | steps/epoch=884 | total=8840 | warmup=884 | loss=ce | alpha_siglo=0.1
[A-xlmr] ep 1/10 step 50/8840 | loss 3.5005 | lr 1.13e-06 | gnorm 3.07 | 51 smp/s | ETA 91m34s
[A-xlmr] ep 1/10 step 100/8840 | loss 3.4997 | lr 2.26e-06 | gnorm 3.41 | 52 smp/s | ETA 89m57s
[A-xlmr] ep 1/10 step 150/8840 | loss 3.4853 | lr 3.39e-06 | gnorm 4.73 | 51 smp/s | ETA 89m43s
[A-xlmr] ep 1/10 step 200/8840 | loss 3.4589 | lr 4.52e-06 | gnorm 5.40 | 51 smp/s | ETA 89m41s
[A-xlmr] ep 1/10 step 250/8840 | loss 3.4329 | lr 5.66e-06 | gnorm 7.10 | 50 smp/s | ETA 89m48s
[A-xlmr] ep 1/10 step 300/8840 | loss 3.3890 | lr 6.79e-06 | gnorm 8.12 | 49 smp/s | ETA 89m52s
[A-xlmr] ep 1/10 step 350/8840 | loss 3.3341 | lr 7.92e-06 | gnorm 13.34 | 48 smp/s | ETA 89m57s
[A-xlmr] ep 1/10 step 400/8840 | loss 3.2792 | lr 9.05e-06 | gnorm 11.25 | 49 smp/s | ETA 89m50s
[A-xlmr] ep 1/10 step 450/8840 | loss 3.2347 | lr 1.02e-05 | gnorm 6.91 | 49 smp/s | ETA 89m38s
[A-xlmr] ep 1/10 step 500/8840 | loss 3

## 3. Modelo B — BETO multi-task (fp16)

In [6]:
tok_b = AutoTokenizer.from_pretrained(MODEL_NAME_B, use_fast=True)
model_b = dl_utils.MultiTaskClassifier(MODEL_NAME_B, use_century=True, dropout=DROPOUT)
print("B:", MODEL_NAME_B, "|", round(sum(p.numel() for p in model_b.parameters())/1e6, 1), "M")


config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertModel LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on 

B: dccuchile/bert-base-spanish-wwm-cased | 109.9 M


In [7]:
%%time
model_b, hist_b, f1_b = dl_utils.train_model(
    model_b, texts[idx_tr], y_dec[idx_tr], y_cen[idx_tr],
    texts[idx_va], y_dec[idx_va], y_cen[idx_va],
    tok_b, cfg=cfg_b, augment_fn=AUG, label="B-beto",
)
print("\nBest val F1 (B):", round(f1_b, 4))


[B-beto] precisión=fp16 | steps/epoch=884 | total=8840 | warmup=884 | loss=ce | alpha_siglo=0.1


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

[B-beto] ep 1/10 step 50/8840 | loss 3.4883 | lr 1.13e-06 | gnorm 3.43 | 53 smp/s | ETA 88m22s
[B-beto] ep 1/10 step 100/8840 | loss 3.4774 | lr 2.26e-06 | gnorm 3.41 | 55 smp/s | ETA 86m21s
[B-beto] ep 1/10 step 150/8840 | loss 3.4533 | lr 3.39e-06 | gnorm 3.49 | 56 smp/s | ETA 84m45s
[B-beto] ep 1/10 step 200/8840 | loss 3.4229 | lr 4.52e-06 | gnorm 4.45 | 56 smp/s | ETA 83m57s
[B-beto] ep 1/10 step 250/8840 | loss 3.3858 | lr 5.66e-06 | gnorm 5.82 | 56 smp/s | ETA 83m16s
[B-beto] ep 1/10 step 300/8840 | loss 3.3444 | lr 6.79e-06 | gnorm 6.35 | 56 smp/s | ETA 82m35s
[B-beto] ep 1/10 step 350/8840 | loss 3.2905 | lr 7.92e-06 | gnorm 7.29 | 56 smp/s | ETA 81m58s
[B-beto] ep 1/10 step 400/8840 | loss 3.2397 | lr 9.05e-06 | gnorm 7.95 | 56 smp/s | ETA 81m24s
[B-beto] ep 1/10 step 450/8840 | loss 3.1905 | lr 1.02e-05 | gnorm 7.56 | 56 smp/s | ETA 80m52s
[B-beto] ep 1/10 step 500/8840 | loss 3.1459 | lr 1.13e-05 | gnorm 9.55 | 56 smp/s | ETA 80m21s
[B-beto] ep 1/10 step 550/8840 | loss 3.1

## 4. Ensemble en holdout (promedio de probabilidades, pesos ∝ F1)

In [8]:
val_texts = list(texts[idx_va])
pa_val = dl_utils.predict_logits_sliding(model_a, tok_a, val_texts, MAX_LENGTH, return_proba=True)
pb_val = dl_utils.predict_logits_sliding(model_b, tok_b, val_texts, MAX_LENGTH, return_proba=True)

W = dl_utils.f1_weights([f1_a, f1_b])
ens_val = dl_utils.combine_proba([pa_val, pb_val], W)
val_metrics_ens = utils.compute_metrics(y_dec[idx_va], ens_val.argmax(axis=1))

print("Pesos ensemble  A/B:", [round(x, 3) for x in W])
utils.print_metrics(utils.compute_metrics(y_dec[idx_va], pa_val.argmax(1)), "A solo")
utils.print_metrics(utils.compute_metrics(y_dec[idx_va], pb_val.argmax(1)), "B solo")
utils.print_metrics(val_metrics_ens, "ENSEMBLE")


Token indices sequence length is longer than the specified maximum sequence length for this model (531 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (568 > 512). Running this sequence through the model will result in indexing errors


Pesos ensemble  A/B: [0.488, 0.512]
[A solo] F1-macro         : 0.2825
[A solo] Accuracy         : 0.2862
[A solo] Mean dist (dec.) : 2.97
[A solo] <=1 decade       : 48.4%
[A solo] <=2 decades      : 62.9%
[A solo] Inter-century err: 21.6%
[B solo] F1-macro         : 0.2908
[B solo] Accuracy         : 0.2900
[B solo] Mean dist (dec.) : 3.02
[B solo] <=1 decade       : 46.7%
[B solo] <=2 decades      : 61.7%
[B solo] Inter-century err: 22.3%
[ENSEMBLE] F1-macro         : 0.3085
[ENSEMBLE] Accuracy         : 0.3107
[ENSEMBLE] Mean dist (dec.) : 2.83
[ENSEMBLE] <=1 decade       : 49.5%
[ENSEMBLE] <=2 decades      : 64.3%
[ENSEMBLE] Inter-century err: 21.1%


## 5. Predicción final + submission

In [9]:
eval_texts = corpus.eval_["text_clean"].tolist()
pa_ev = dl_utils.predict_logits_sliding(model_a, tok_a, eval_texts, MAX_LENGTH, return_proba=True)
pb_ev = dl_utils.predict_logits_sliding(model_b, tok_b, eval_texts, MAX_LENGTH, return_proba=True)
ens_ev = dl_utils.combine_proba([pa_ev, pb_ev], W)

pred_decade = np.array([utils.IDX_TO_DECADE[i] for i in ens_ev.argmax(axis=1)])
sub_path = utils.write_submission(corpus.eval_["id"].values, pred_decade, f"{OUT_DIR}/submission_07.csv")
utils.write_submission(corpus.eval_["id"].values, pred_decade, f"{OUT_DIR}/submission.csv")
print("Final submission ->", sub_path, "| décadas únicas:", len(set(pred_decade)))


Final submission -> /kaggle/working/submission_07.csv | décadas únicas: 39


## 6. Guardado del bundle (formato uniforme para NB08)

In [10]:
import joblib

MODEL_A_DIR = f"{OUT_DIR}/modelo_v07_A_xlmr"
MODEL_B_DIR = f"{OUT_DIR}/modelo_v07_B_beto"
dl_utils.save_multitask(model_a, tok_a, MODEL_A_DIR)
dl_utils.save_multitask(model_b, tok_b, MODEL_B_DIR)

payload = {
    "version": "07_multitask_external_ensemble",
    "kind": "multitask_ensemble",
    "models": [
        {"name": MODEL_NAME_A, "dir": MODEL_A_DIR, "use_century": True, "val_f1": float(f1_a), "weight": float(W[0])},
        {"name": MODEL_NAME_B, "dir": MODEL_B_DIR, "use_century": True, "val_f1": float(f1_b), "weight": float(W[1])},
    ],
    "max_length": MAX_LENGTH,
    "num_decades": utils.NUM_DECADES,
    "num_centuries": utils.NUM_CENTURIES,
    "idx_to_decade": utils.IDX_TO_DECADE,
    "val_f1_macro": float(val_metrics_ens["f1_macro"]),
    "val_metrics": val_metrics_ens,
    "multi_task": {"alpha_century": ALPHA_CENTURY},
    "regularization": {"dropout": DROPOUT, "loss_kind": LOSS_KIND, "smooth_eps": SMOOTH_EPS},
    "augmentation": {"used": USE_AUGMENT, "p_space": P_SPACE, "p_hyphen": P_HYPHEN, "p_char": P_CHAR},
    "used_external_data": bool(USE_EXTERNAL_DATA),
    "history": {"A": hist_a, "B": hist_b},
    "seed": utils.SEED,
}
joblib.dump(payload, f"{OUT_DIR}/modelo_final.joblib", compress=3)
print("Saved bundle:", f"{OUT_DIR}/modelo_final.joblib")


Saved bundle: /kaggle/working/modelo_final.joblib


### Descarga directa (FileLink) — evita el `Save Version` que mata la sesión

In [11]:
from IPython.display import FileLink, display

for name in ("submission_07.csv", "submission.csv", "modelo_final.joblib"):
    path = f"{OUT_DIR}/{name}"
    if os.path.exists(path):
        print(f"{name} ({os.path.getsize(path)/1e6:.1f} MB)")
        display(FileLink(path))


submission_07.csv (0.0 MB)


/kaggle/working/submission_07.csv

submission.csv (0.0 MB)


/kaggle/working/submission.csv

modelo_final.joblib (0.0 MB)


/kaggle/working/modelo_final.joblib

## 7. Iteraciones (rúbrica §5.2.1)

| # | Cambio principal | F1-macro |
|---|---|---|
| Legacy E1 | TF-IDF char_wb+word + stacking | 0.296 (Kaggle) |
| 06 | + label smoothing gaussiano | 0.20 (val) — falsada |
| 09 | XLM-R + CE plana, 12 epochs | **0.288 val / 0.29130 Kaggle** |
| **07** | **+ ensemble XLM-R + BETO + multi-task siglo** | **REPORTAR** |

**Aprendido de NB09:** (1) el fix de smoothing está cuantificado (+0.09 vs NB06); (2) val≈Kaggle
(0.288 vs 0.291) → el holdout es fiable, se itera sin quemar submissions; (3) **overfitting**:
train_loss seguía cayendo pero val F1 se aplanó en ~ep10 → el cuello de botella es generalización,
no entrenamiento. Por eso 07 ataca *varianza*: ensemble (promedia errores) + multi-task siglo
(el error inter-siglo quedó en ~21%). Las palancas `DROPOUT`/`USE_UNIFORM_SMOOTH` quedan listas
para el ablation si tras el ensemble el holdout sigue plano.

**Decisiones validadas:** XLM-R (multilingüe, cubre latín/italiano/catalán) + BETO (español);
CE plana > smoothing gaussiano; modelos vigentes (bertin/BNE descartados, mDeBERTa inestable).
**Próximo:** NB08 (híbrido legacy — la palanca de mayor valor) y NB10 (+mDeBERTa best-effort).
